In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import time
import numpy as np
import pandas as pd
import torch.nn.functional as F

from notebooks.local.utils import get_paths, create_folders, safe_cosine_similarity

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

create_folders(PATHS)
print("Device:", DEVICE)


In [ ]:
from notebooks.local.utils import get_paths, create_folders

PATHS    = get_paths()
create_folders(PATHS)

import os
ap10k_images = os.path.join(PATHS['ap10k'], 'images')
ap10k_ann    = os.path.join(PATHS['ap10k'], 'annotations')

if not os.path.exists(ap10k_images) or not os.path.exists(ap10k_ann):
    print("AP-10K dataset not found at:", PATHS['ap10k'])
    print("Download it manually:")
    print("  1. Visit https://github.com/AlexTheBad/AP-10K")
    print("  2. Download via the Google Drive link in their README")
    print("  3. Extract so data/AP-10K/images/ and data/AP-10K/annotations/ exist")
    raise FileNotFoundError("AP-10K not present. See instructions above.")
else:
    print("AP-10K present:", PATHS['ap10k'])

if not os.path.exists(PATHS['dinov2_w']):
    from notebooks.local.utils import download_file
    print("Downloading DINOv2 ViT-B/14 weights (~330 MB) ...")
    download_file(
        "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth",
        PATHS['dinov2_w'], desc="DINOv2",
    )
else:
    print("DINOv2 weights present.")

if not os.path.exists(PATHS['sam_w']):
    from notebooks.local.utils import download_file
    print("Downloading SAM ViT-B weights (~370 MB) ...")
    download_file(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        PATHS['sam_w'], desc="SAM",
    )
else:
    print("SAM weights present.")

if not os.path.exists(PATHS['dinov3_w']):
    print("WARNING: DINOv3 weights not found at", PATHS['dinov3_w'])
    print("  Place dinov3_vitb16_pretrain.pth in weights/ (from project maintainer).")
else:
    print("DINOv3 weights present.")


In [ ]:
from src.datasets.ap10k_dataset import AP10KDataset
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.models.segment_anything.segment_anything import sam_model_registry
from src.features.extractor import (
    extract_dense_features, extract_dense_features_SAM,
    pixel_to_patch_coord, patch_to_pixel_coord,
)
from src.matching.strategies import find_best_match_argmax
from src.metrics.pck import compute_pck_ap10k

ap10k_test = AP10KDataset(PATHS['ap10k'], split='test', split_num=1, min_visible_kps=4)
print(f"AP-10K test pairs: {len(ap10k_test)}")


def load_model(backbone, paths, device, use_fp16=False):
    ft_map = {
        'dinov2': (paths.get('dinov2_ft'), paths['dinov2_w']),
        'dinov3': (paths.get('dinov3_ft'), paths['dinov3_w']),
        'sam':    (paths.get('sam_ft'),    paths['sam_w']),
    }
    ft_path, base_path = ft_map[backbone]

    if backbone == 'dinov2':
        model = vit_base_v2(img_size=(518,518), patch_size=14,
                            num_register_tokens=0, block_chunks=0, init_values=1.0)
        img_size, patch_size = 518, 14
    elif backbone == 'dinov3':
        model = vit_base_v3(img_size=512, patch_size=16, n_storage_tokens=4, mask_k_bias=True, layerscale_init=1.0e-05, norm_layer="layernormbf16")
        img_size, patch_size = 512, 16
    elif backbone == 'sam':
        model = sam_model_registry['vit_b'](checkpoint=base_path)
        img_size, patch_size = 512, 16

    if backbone != 'sam':
        if ft_path and os.path.exists(ft_path):
            ckpt = torch.load(ft_path, map_location=device, weights_only=True)
            state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
            model.load_state_dict(state, strict=True)
            print(f"Loaded fine-tuned {backbone}")
        else:
            ckpt = torch.load(base_path, map_location=device, weights_only=True)
            model.load_state_dict(ckpt, strict=True)
            print(f"Loaded pretrained {backbone}")

    model = model.to(device)
    if use_fp16 and backbone != 'sam':
        model = model.half()
    model.eval()
    return model, img_size, patch_size


## DINOv2 on AP-10K

In [ ]:
out_dir = os.path.join(PATHS['step4_ap10k'], 'dinov2')
os.makedirs(out_dir, exist_ok=True)
stats_path = os.path.join(out_dir, 'overall_stats.json')

if os.path.exists(stats_path):
    print("DINOv2 AP-10K — already done.")
    with open(stats_path) as f:
        dinov2_ap10k = json.load(f)
    print(dinov2_ap10k)
else:
    model, img_size, patch_size = load_model('dinov2', PATHS, DEVICE, USE_FP16)
    t0 = time.time()
    per_img = []

    with torch.no_grad():
        for idx, sample in enumerate(ap10k_test):
            src_t = sample['src_img'].unsqueeze(0).to(DEVICE)
            tgt_t = sample['trg_img'].unsqueeze(0).to(DEVICE)
            if USE_FP16:
                src_t, tgt_t = src_t.half(), tgt_t.half()

            src_t = F.interpolate(src_t, size=(img_size, img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(tgt_t, size=(img_size, img_size), mode='bilinear', align_corners=False)

            src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_orig = (sample['trg_imsize'][2], sample['trg_imsize'][1])

            src_feat = extract_dense_features(model, src_t)
            tgt_feat = extract_dense_features(model, tgt_t)
            _, H, W, D = tgt_feat.shape
            tgt_flat = tgt_feat.reshape(H * W, D)

            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            pred = []

            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_orig, patch_size, img_size)
                sf = src_feat[0, py, px, :]
                sims = safe_cosine_similarity(sf.float(), tgt_flat.float())
                mx, my = find_best_match_argmax(sims, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_orig, patch_size, img_size)
                pred.append([rx, ry])

            image_pcks = {}
            for thr in THRESHOLDS:
                pck, _, _ = compute_pck_ap10k(pred, trg_kps, tgt_orig, thr)
                image_pcks[thr] = pck
            per_img.append({'pck_scores': image_pcks})
            if (idx + 1) % 200 == 0:
                print(f"  {idx+1}/{len(ap10k_test)}")

    elapsed = time.time() - t0
    dinov2_ap10k = {"inference_time_sec": elapsed}
    for thr in THRESHOLDS:
        vals = [m['pck_scores'][thr] for m in per_img]
        dinov2_ap10k[f"pck@{thr:.2f}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        print(f"DINOv2 AP-10K PCK@{thr:.2f}: {np.mean(vals):.2f}%")
    with open(stats_path, 'w') as f:
        json.dump(dinov2_ap10k, f, indent=2)
    del model; torch.cuda.empty_cache()
    print("DINOv2 AP-10K done.")


## DINOv3 on AP-10K

In [ ]:
out_dir = os.path.join(PATHS['step4_ap10k'], 'dinov3')
os.makedirs(out_dir, exist_ok=True)
stats_path = os.path.join(out_dir, 'overall_stats.json')

if os.path.exists(stats_path):
    print("DINOv3 AP-10K — already done.")
    with open(stats_path) as f:
        dinov3_ap10k = json.load(f)
    print(dinov3_ap10k)
else:
    model, img_size, patch_size = load_model('dinov3', PATHS, DEVICE, USE_FP16)
    t0 = time.time()
    per_img = []

    with torch.no_grad():
        for idx, sample in enumerate(ap10k_test):
            src_t = sample['src_img'].unsqueeze(0).to(DEVICE)
            tgt_t = sample['trg_img'].unsqueeze(0).to(DEVICE)
            if USE_FP16:
                src_t, tgt_t = src_t.half(), tgt_t.half()
            src_t = F.interpolate(src_t, size=(img_size,img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(tgt_t, size=(img_size,img_size), mode='bilinear', align_corners=False)

            src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_orig = (sample['trg_imsize'][2], sample['trg_imsize'][1])

            src_feat = extract_dense_features(model, src_t)
            tgt_feat = extract_dense_features(model, tgt_t)
            _, H, W, D = tgt_feat.shape
            tgt_flat = tgt_feat.reshape(H * W, D)

            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            pred = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_orig, patch_size, img_size)
                sf = src_feat[0, py, px, :]
                sims = safe_cosine_similarity(sf.float(), tgt_flat.float())
                mx, my = find_best_match_argmax(sims, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_orig, patch_size, img_size)
                pred.append([rx, ry])

            image_pcks = {}
            for thr in THRESHOLDS:
                pck, _, _ = compute_pck_ap10k(pred, trg_kps, tgt_orig, thr)
                image_pcks[thr] = pck
            per_img.append({'pck_scores': image_pcks})
            if (idx + 1) % 200 == 0:
                print(f"  {idx+1}/{len(ap10k_test)}")

    elapsed = time.time() - t0
    dinov3_ap10k = {"inference_time_sec": elapsed}
    for thr in THRESHOLDS:
        vals = [m['pck_scores'][thr] for m in per_img]
        dinov3_ap10k[f"pck@{thr:.2f}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        print(f"DINOv3 AP-10K PCK@{thr:.2f}: {np.mean(vals):.2f}%")
    with open(stats_path, 'w') as f:
        json.dump(dinov3_ap10k, f, indent=2)
    del model; torch.cuda.empty_cache()
    print("DINOv3 AP-10K done.")


## SAM on AP-10K

In [ ]:
out_dir = os.path.join(PATHS['step4_ap10k'], 'sam')
os.makedirs(out_dir, exist_ok=True)
stats_path = os.path.join(out_dir, 'overall_stats.json')

if os.path.exists(stats_path):
    print("SAM AP-10K — already done.")
    with open(stats_path) as f:
        sam_ap10k = json.load(f)
    print(sam_ap10k)
else:
    model, img_size, patch_size = load_model('sam', PATHS, DEVICE, USE_FP16)
    t0 = time.time()
    per_img = []

    with torch.no_grad():
        for idx, sample in enumerate(ap10k_test):
            src_t = sample['src_img'].unsqueeze(0).to(DEVICE)
            tgt_t = sample['trg_img'].unsqueeze(0).to(DEVICE)

            src_orig = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_orig = (sample['trg_imsize'][2], sample['trg_imsize'][1])

            src_feat = extract_dense_features_SAM(model, src_t, image_size=img_size)
            tgt_feat = extract_dense_features_SAM(model, tgt_t, image_size=img_size)
            _, H, W, D = tgt_feat.shape
            tgt_flat = tgt_feat.reshape(H * W, D)

            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            pred = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_orig, patch_size, img_size)
                sf = src_feat[0, py, px, :]
                sims = safe_cosine_similarity(sf.float(), tgt_flat.float())
                mx, my = find_best_match_argmax(sims, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_orig, patch_size, img_size)
                pred.append([rx, ry])

            image_pcks = {}
            for thr in THRESHOLDS:
                pck, _, _ = compute_pck_ap10k(pred, trg_kps, tgt_orig, thr)
                image_pcks[thr] = pck
            per_img.append({'pck_scores': image_pcks})
            if (idx + 1) % 200 == 0:
                print(f"  {idx+1}/{len(ap10k_test)}")

    elapsed = time.time() - t0
    sam_ap10k = {"inference_time_sec": elapsed}
    for thr in THRESHOLDS:
        vals = [m['pck_scores'][thr] for m in per_img]
        sam_ap10k[f"pck@{thr:.2f}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        print(f"SAM AP-10K PCK@{thr:.2f}: {np.mean(vals):.2f}%")
    with open(stats_path, 'w') as f:
        json.dump(sam_ap10k, f, indent=2)
    del model; torch.cuda.empty_cache()
    print("SAM AP-10K done.")


## Summary

In [ ]:
rows = []
for label, d in [
    ('DINOv2', os.path.join(PATHS['step4_ap10k'], 'dinov2')),
    ('DINOv3', os.path.join(PATHS['step4_ap10k'], 'dinov3')),
    ('SAM',    os.path.join(PATHS['step4_ap10k'], 'sam')),
]:
    stats_path = os.path.join(d, 'overall_stats.json')
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            s = json.load(f)
        rows.append({
            'Model': label,
            'PCK@0.05 (diag)': round(s.get('pck@0.05',{}).get('mean', float('nan')), 2),
            'PCK@0.10 (diag)': round(s.get('pck@0.10',{}).get('mean', float('nan')), 2),
            'PCK@0.20 (diag)': round(s.get('pck@0.20',{}).get('mean', float('nan')), 2),
        })
    else:
        rows.append({'Model': label, 'PCK@0.05 (diag)': '-', 'PCK@0.10 (diag)': '-', 'PCK@0.20 (diag)': '-'})

df = pd.DataFrame(rows).set_index('Model')
print("AP-10K Results (PCK normalized by image diagonal):")
print(df.to_string())
